In [6]:
from transformers import TFBertModel, AutoTokenizer
import pandas as pd

2025-04-26 19:16:44.750667: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Model Setup
We load a pre-trained BERT model and it's tokenizer.

In [7]:
#specify the pre-trained model to use: BERT-base-cased
model_name = 'bert-base-cased'

In [9]:
#Instantiate the model and tokenizer for the specified pre-trained model
model = TFBertModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer

2025-04-26 19:19:57.245427: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some layers from the model checkpoint at bert-base-cased were not used when initializing TFBertModel: ['nsp___cls', 'mlm___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertModel were initialized from the model check

BertTokenizerFast(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'})

In [10]:
#set a sentence for analysis
sentence = "When life give you lemons, don't make lemonade."

In [11]:
#Tokenize the sentence
tokens = tokenizer.tokenize(sentence)
tokens

['When',
 'life',
 'give',
 'you',
 'lemon',
 '##s',
 ',',
 'don',
 "'",
 't',
 'make',
 'lemon',
 '##ade',
 '.']

# Vocabulary and Token IDs
we create a DataFrame with the tokenizer's vocabulary

In [12]:
# create a DataFrame with the tokenizer's vocabulary
vocab = tokenizer.vocab
vocab_df = pd.DataFrame({"token": vocab.keys(), "token_id": vocab.values()})
vocab_df = vocab_df.sort_values(by="token_id").set_index("token_id")

vocab_df

,token
token_id,
0,[PAD]
1,[unused1]
2,[unused2]
3,[unused3]
4,[unused4]
...,...
28991,##）
28992,##，
28993,##－


# Encoding and Decoding
Encode the sentence into IDs and then decode it back to text.

In [14]:
#Encode the sentence into token_ids using the tokenizer
token_ids = tokenizer.encode(sentence)
token_ids

[101,
 1332,
 1297,
 1660,
 1128,
 22782,
 1116,
 117,
 1274,
 112,
 189,
 1294,
 22782,
 6397,
 119,
 102]

# Compare Token Lengths
compare the length of tokens and token IDs

In [15]:
#print the length of tokens and token_ids
print("Number of tokens: ", len(tokens))
print("Number of token IDs:", len(token_ids))

Number of tokens:  14
Number of token IDs: 16


# Explore Token Data
Look at specific tokens by their IDs.

In [16]:
#Access the tokens in the vocabulary DataFrame by index
print("Token at position 101: ", vocab_df.iloc[101])
print("Token at position 102: ", vocab_df.iloc[102])

Token at position 101:  token    [CLS]
Name: 101, dtype: object
Token at position 102:  token    [SEP]
Name: 102, dtype: object


# Token and ID Pairing
show pairs of tokens and their IDs.

In [17]:
#zip tokens and token_ids (excluding the first token_ids for [CLS] and [SEP])
list(zip(tokens, token_ids[1:-1]))

[('When', 1332),
 ('life', 1297),
 ('give', 1660),
 ('you', 1128),
 ('lemon', 22782),
 ('##s', 1116),
 (',', 117),
 ('don', 1274),
 ("'", 112),
 ('t', 189),
 ('make', 1294),
 ('lemon', 22782),
 ('##ade', 6397),
 ('.', 119)]

In [20]:
#Decode the token_ids (excluding the first and last token_ids for [CLS] and [SEP] back into the original sentence)
tokenizer.decode(token_ids[1:-1])

"When life give you lemons, don't make lemonade."

In [21]:
#Tokenize the sentence using tokenizer's '__call__' method
tokenizer_out = tokenizer(sentence)
tokenizer_out

{'input_ids': [101, 1332, 1297, 1660, 1128, 22782, 1116, 117, 1274, 112, 189, 1294, 22782, 6397, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

# Handling Multiple Sentences
Tokenize two sentences with and without padding, and decode them.

In [22]:
#Create a new sentence by removing "don't " from the original sentence
sentence2 = sentence.replace("don't ", "")
sentence2

'When life give you lemons, make lemonade.'

In [23]:
#Tokenize both sentences with padding
tokenizer_out2 = tokenizer([sentence, sentence2], padding=True)
tokenizer_out2

{'input_ids': [[101, 1332, 1297, 1660, 1128, 22782, 1116, 117, 1274, 112, 189, 1294, 22782, 6397, 119, 102], [101, 1332, 1297, 1660, 1128, 22782, 1116, 117, 1294, 22782, 6397, 119, 102, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]]}

In [24]:
#Decode the tokenized input_ids for both sentences
tokenizer.decode(tokenizer_out2["input_ids"][0])

"[CLS] When life give you lemons, don't make lemonade. [SEP]"

In [25]:
tokenizer.decode(tokenizer_out2["input_ids"][1])

'[CLS] When life give you lemons, make lemonade. [SEP] [PAD] [PAD] [PAD]'

# Conclusion
This notebook walked you through how to use a BERT tokenizer to process text, turning it into tokens and IDs, and how to handle multiple sentences.